## Tahap Persiapan : Inisialisasi SparkSession & Import Library


In [1]:
import time, json
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (SparkSession.builder
    .appName('TidarMart_2505060024')
    .master('local[2]')
    .config('spark.driver.memory', '1536m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())

HDFS_BASE = "hdfs://localhost:9000/tidarmart"
SEED = 60024
print("Spark UI:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/28 12:04:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://zeroone:4040


## Tahap 1 : Penyimpanan Terdistribusi & Ingesti (HDFS)
• Buat struktur direktori di HDFS, misalnya /tidarmart/raw/transaksi, /tidarmart/raw/pelanggan,
/tidarmart/raw/produk, /tidarmart/stream_input, /tidarmart/curated, /tidarmart/models.

• Unggah seluruh data ke HDFS. Tampilkan bukti ukuran, jumlah blok, dan faktor replikasi (hdfs dfs -du -
h, hdfs fsck).

• Baca seluruh data dari HDFS (hdfs://localhost:9000/...) bukan dari path lokal pada tahap-tahap berikutnya.

• Jelaskan: mengapa data mentah dan data bersih dipisah direktori, dan apa arti replikasi = 1 pada VM
pseudo-distributed dibanding klaster nyata

In [2]:
# Membuat Direktori & Mengunggah Data ke HDFS
%%bash
hdfs dfs -mkdir -p /tidarmart/raw/transaksi /tidarmart/raw/pelanggan /tidarmart/raw/produk /tidarmart/raw/stream_input /tidarmart/curated /tidarmart/models
hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/

In [4]:
# Verifikasi Ukuran, Blok, dan Replikasi (Bukti Volume)

!hdfs dfs -du -h /tidarmart/raw/
!hdfs fsck /tidarmart/raw/transaksi -files -blocks

6.0 M    6.0 M    /tidarmart/raw/pelanggan
685.3 K  685.3 K  /tidarmart/raw/produk
0        0        /tidarmart/raw/stream_input
187.2 M  187.2 M  /tidarmart/raw/transaksi
Connecting to namenode via http://localhost:9870/fsck?ugi=zerouno&files=1&blocks=1&path=%2Ftidarmart%2Fraw%2Ftransaksi
FSCK started by zerouno (auth:SIMPLE) from /127.0.0.1 for path /tidarmart/raw/transaksi at Thu Nov 26 03:42:41 UTC 2026

/tidarmart/raw/transaksi <dir>
/tidarmart/raw/transaksi/transaksi_part1.csv 49081412 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1327634461-10.0.2.15-1788789334784:blk_1073741866_1042 len=49081412 Live_repl=1

/tidarmart/raw/transaksi/transaksi_part2.csv 49081080 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1327634461-10.0.2.15-1788789334784:blk_1073741867_1043 len=49081080 Live_repl=1

/tidarmart/raw/transaksi/transaksi_part3.csv 49077294 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1327634461-10.0.2.15-1788789334784:blk_1073741868_1044 len=490772

## Tahap 2 : Profiling Kualitas Data (Veracity & Variety)

• Hitung jumlah baris, jumlah order_id unik, persentase nilai kosong per kolom, jumlah harga negatif,
distribusi qty (temukan outlier dengan kuantil, jenis nya bebas), dan variasi penulisan kota.

• Sajikan hasil profiling dalam satu tabel ringkasan “sebelum pembersihan”.

• Untuk profil pelanggan dan produk: tunjukkan bahwa atribut produk bervariasi menurut kategori (berapa
persen produk memiliki setiap atribut) dan bagaimana field tags diperlakukan.

In [2]:
# Membaca Data dari HDFS dengan Skema Eksplisit untuk JSON
df_trx_raw = spark.read.csv(f"{HDFS_BASE}/raw/transaksi/*.csv", header=True, inferSchema=True)

schema_pelanggan = StructType([
    StructField("pelanggan_id", IntegerType()), StructField("nama", StringType()),
    StructField("kota", StringType()), StructField("tanggal_daftar", StringType()),
    StructField("membership", StringType())
])
schema_produk = StructType([
    StructField("produk_id", IntegerType()), StructField("nama_produk", StringType()),
    StructField("kategori", StringType()), StructField("harga_dasar", LongType()),
    StructField("berat_kg", DoubleType()), StructField("garansi_bulan", IntegerType()),
    StructField("voltase", IntegerType()), StructField("ukuran", StringType()),
    StructField("bahan", StringType()), StructField("kadaluarsa_hari", IntegerType()),
    StructField("penerbit", StringType()), StructField("tags", ArrayType(StringType()))
])

df_pelanggan = spark.read.schema(schema_pelanggan).json(f"{HDFS_BASE}/raw/pelanggan/pelanggan.json")
df_produk = spark.read.schema(schema_produk).json(f"{HDFS_BASE}/raw/produk/produk.json")

In [3]:
# Profiling Transaksi
total_rows = df_trx_raw.count()
unique_orders = df_trx_raw.select("order_id").distinct().count()
neg_harga = df_trx_raw.filter(F.col("harga_satuan") < 0).count()
outlier_qty = df_trx_raw.filter(F.col("qty") > 50).count()
qty_q = df_trx_raw.approxQuantile("qty", [0.5, 0.99, 0.999, 1.0], 0.001)

null_qty = df_trx_raw.filter(F.col("qty").isNull() | F.isnan("qty")).count()
null_harga = df_trx_raw.filter(F.col("harga_satuan").isNull() | F.isnan("harga_satuan")).count()
variasi_kota = df_trx_raw.select("kota").distinct().count()

profil_sebelum = pd.DataFrame({
    "Metrik": ["Total Baris", "Order Unik", "Duplikat", "Null Qty (%)", "Null Harga (%)", "Harga Negatif", "Outlier Qty (>50)", "Variasi Nama Kota"],
    "Sebelum": [total_rows, unique_orders, total_rows - unique_orders, round(null_qty/total_rows*100, 3), round(null_harga/total_rows*100, 3), neg_harga, outlier_qty, variasi_kota]
})
print("Kuantil Qty [50%, 99%, 99.9%, 100%]:", qty_q)
display(profil_sebelum)

[Stage 24:>                                                         (0 + 2) / 2]

Kuantil Qty [50%, 99%, 99.9%, 100%]: [4.0, 7.0, 999.0, 999.0]


,Metrik,Sebelum
0,Total Baris,2010000.000
1,Order Unik,2000000.000
2,Duplikat,10000.000
3,Null Qty (%),0.299
4,Null Harga (%),0.300
5,Harga Negatif,2005.000
6,Outlier Qty (>50),1035.000
7,Variasi Nama Kota,24.000


In [4]:
# Profiling Produk & Pelanggan
total_prod = df_produk.count()
print("Persentase Kelengkapan Atribut Produk (%):")
df_produk.select([
    F.round(F.count(F.col(c)) / total_prod * 100, 2).alias(c) for c in df_produk.columns
]).show()

# Cek penanganan field array opsional (tags)
df_produk.select("produk_id", "kategori", "tags").filter(F.col("tags").isNotNull()).show(5, truncate=False)

Persentase Kelengkapan Atribut Produk (%):
+---------+-----------+--------+-----------+--------+-------------+-------+------+-----+---------------+--------+-----+
|produk_id|nama_produk|kategori|harga_dasar|berat_kg|garansi_bulan|voltase|ukuran|bahan|kadaluarsa_hari|penerbit| tags|
+---------+-----------+--------+-----------+--------+-------------+-------+------+-----+---------------+--------+-----+
|    100.0|      100.0|   100.0|      100.0|   100.0|        12.52|  12.52| 13.68|13.68|          12.04|   11.82|30.52|
+---------+-----------+--------+-----------+--------+-------------+-------+------+-----+---------------+--------+-----+

+---------+---------+------------------------+
|produk_id|kategori |tags                    |
+---------+---------+------------------------+
|3        |Buku     |[terlaris, lokal, promo]|
|6        |Kesehatan|[promo]                 |
|11       |Fashion  |[promo, lokal]          |
|12       |Makanan  |[terlaris]              |
|13       |Olahraga |[lokal

## Tahap 3 : ETL & Penyimpanan Kolumnar

• Bersihkan transaksi sesuai temuan Tahap 2: deduplikasi, penanganan nilai kosong, harga negatif, outlier
qty, normalisasi nama kota. Setiap aturan harus disertai alasan (dibuang, diisi, atau dikoreksi, dan
mengapa).

• Tambahkan kolom turunan: total_nilai (qty × harga_satuan × (1 − diskon/100)), bulan,
hari_dalam_minggu, flag_batal.

• Simpan hasil ke HDFS /tidarmart/curated dalam format Parquet dengan partitionBy yang Anda pilih dan
justifikasikan (misalnya kota atau bulan).

• Bandingkan CSV dan Parquet: ukuran di HDFS dan waktu eksekusi satu query agregasi yang sama.
Sajikan dalam tabel.

• Tampilkan laporan “sesudah pembersihan” berdampingan dengan laporan sebelumnya.

In [9]:
# Pembersihan Data & Penambahan Kolom Turunan
df_clean = (df_trx_raw
    .dropDuplicates(["order_id"])                              
    .dropna(subset=["qty", "harga_satuan"])                    
    .withColumn("harga_satuan", F.abs(F.col("harga_satuan")))  
    .filter((F.col("qty") > 0) & (F.col("qty") <= 50))         
    .withColumn("kota", F.initcap(F.trim(F.col("kota"))))     
    .withColumn("tanggal_ts", F.to_timestamp("tanggal"))
    .withColumn("total_nilai", F.col("qty") * F.col("harga_satuan") * (1 - F.col("diskon_persen") / 100.0))
    .withColumn("bulan", F.date_format("tanggal_ts", "yyyy-MM"))
    .withColumn("hari_dalam_minggu", F.dayofweek("tanggal_ts"))
    .withColumn("flag_batal", F.when(F.col("status_pesanan") == "Dibatalkan", 1).otherwise(0))
)

In [10]:
# Menyimpan ke HDFS Format Parquet (Partisi "bulan")
df_clean.write.mode("overwrite").partitionBy("bulan").parquet(f"{HDFS_BASE}/curated/transaksi_parquet")
!hdfs dfs -du -h /tidarmart/curated/transaksi_parquet

0       0       /tidarmart/curated/transaksi_parquet/_SUCCESS
12.6 M  37.7 M  /tidarmart/curated/transaksi_parquet/bulan=2026-01
11.4 M  34.1 M  /tidarmart/curated/transaksi_parquet/bulan=2026-02
12.6 M  37.8 M  /tidarmart/curated/transaksi_parquet/bulan=2026-03
12.2 M  36.5 M  /tidarmart/curated/transaksi_parquet/bulan=2026-04
12.6 M  37.8 M  /tidarmart/curated/transaksi_parquet/bulan=2026-05
12.1 M  36.4 M  /tidarmart/curated/transaksi_parquet/bulan=2026-06


In [11]:
# Benchmark CSV vs Parquet & Tabel "Sesudah Pembersihan"
df_trx = spark.read.parquet(f"{HDFS_BASE}/curated/transaksi_parquet")

t0 = time.time(); df_trx_raw.groupBy("kota").agg(F.sum("qty")).collect(); t_csv = time.time() - t0
t0 = time.time(); df_trx.groupBy("kota").agg(F.sum("qty")).collect(); t_parq = time.time() - t0
print(f"Waktu Query -> CSV: {t_csv:.2f} dtk | Parquet: {t_parq:.2f} dtk")

# Perbandingan Sebelum vs Sesudah
tot_c = df_trx.count()
profil_sebelum["Sesudah"] = [
    tot_c, df_trx.select("order_id").distinct().count(), 0,
    df_trx.filter(F.col("qty").isNull()).count(), df_trx.filter(F.col("harga_satuan").isNull()).count(),
    df_trx.filter(F.col("harga_satuan") < 0).count(), df_trx.filter(F.col("qty") > 50).count(),
    df_trx.select("kota").distinct().count()
]
display(profil_sebelum)

Waktu Query -> CSV: 4.19 dtk | Parquet: 2.70 dtk


[Stage 68:>                                                         (0 + 2) / 2]

,Metrik,Sebelum,Sesudah
0,Total Baris,2010000.000,1987007
1,Order Unik,2000000.000,1987007
2,Duplikat,10000.000,0
3,Null Qty (%),0.299,0
4,Null Harga (%),0.300,0
5,Harga Negatif,2005.000,0
6,Outlier Qty (>50),1035.000,0
7,Variasi Nama Kota,24.000,8


In [12]:
spark.stop()

## Tahap 4 : Analitik Batch 


In [5]:
# Persiapan Data Bersih & Filter Transaksi Selesai
df_trx = spark.read.parquet(f"{HDFS_BASE}/curated/transaksi_parquet")
df_selesai = df_trx.filter(F.col("status_pesanan") == "Selesai")

In [6]:
# A. Broadcast Join (Omzet per Kategori per Kota)
df_join = (df_selesai
    .join(F.broadcast(df_produk.select("produk_id", "nama_produk", "kategori")), "produk_id")
    .join(F.broadcast(df_pelanggan.select("pelanggan_id", "membership")), "pelanggan_id"))

df_join.groupBy("kota", "kategori").agg(F.sum("total_nilai").alias("omzet")) \
       .orderBy("kota", F.desc("omzet")).show(12)

[Stage 40:>                                                         (0 + 2) / 2]

+--------+------------+--------------+
|    kota|    kategori|         omzet|
+--------+------------+--------------+
| Kebumen|     Fashion| 2.88724855E10|
| Kebumen|  Elektronik|  2.8622165E10|
| Kebumen|Rumah Tangga|2.857138275E10|
| Kebumen|     Makanan| 2.65929035E10|
| Kebumen|    Olahraga| 2.64698755E10|
| Kebumen|   Kesehatan| 2.62071665E10|
| Kebumen|        Buku|2.505357525E10|
| Kebumen|    Otomotif|2.364432775E10|
|Magelang|Rumah Tangga| 3.01374845E10|
|Magelang|  Elektronik|2.983639175E10|
|Magelang|     Fashion|  2.9486142E10|
|Magelang|     Makanan|  2.8221333E10|
+--------+------------+--------------+
only showing top 12 rows



In [7]:
# B. Tren Omzet Bulanan & Pertumbuhan (lag) + Grafik
w_bulan = Window.orderBy("bulan")
df_bulanan = (df_selesai.groupBy("bulan").agg(F.sum("total_nilai").alias("omzet"))
    .withColumn("omzet_prev", F.lag("omzet").over(w_bulan))
    .withColumn("growth_pct", F.round((F.col("omzet") - F.col("omzet_prev")) / F.col("omzet_prev") * 100, 2)))
df_bulanan.show()

pdf_b = df_bulanan.toPandas()
plt.figure(figsize=(7, 3))
plt.plot(pdf_b["bulan"], pdf_b["omzet"] / 1e9, marker="o")
plt.title("Tren Omzet Bulanan TidarMart (Miliar Rp)")
plt.ylabel("Omzet (Miliar Rp)"); plt.grid(True); plt.show()

26/11/28 12:14:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 1

+-------+---------------+---------------+----------+
|  bulan|          omzet|     omzet_prev|growth_pct|
+-------+---------------+---------------+----------+
|2026-01|3.0074566725E11|           NULL|      NULL|
|2026-02|2.7197526675E11|3.0074566725E11|     -9.57|
|2026-03|3.0127041575E11|2.7197526675E11|     10.77|
|2026-04|2.9058154625E11|3.0127041575E11|     -3.55|
|2026-05|  3.00722052E11|2.9058154625E11|      3.49|
|2026-06|2.9155883375E11|  3.00722052E11|     -3.05|
+-------+---------------+---------------+----------+



26/11/28 12:14:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:14:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 1

<Figure size 700x300 with 1 Axes>

In [8]:
# C: Top 5 Produk Terlaris per Kota (dense_rank)
w_kota = Window.partitionBy("kota").orderBy(F.desc("omzet"))
df_join.groupBy("kota", "nama_produk").agg(F.sum("total_nilai").alias("omzet")) \
       .withColumn("rank", F.dense_rank().over(w_kota)) \
       .filter(F.col("rank") <= 5).orderBy("kota", "rank").show(15)

[Stage 57:>                                                         (0 + 2) / 2]

+---------+-----------------+---------+----+
|     kota|      nama_produk|    omzet|rank|
+---------+-----------------+---------+----+
|  Kebumen|     Fashion-2884|3.24375E8|   1|
|  Kebumen|  Elektronik-4615| 3.1455E8|   2|
|  Kebumen| Rumah Tangga-807|  2.997E8|   3|
|  Kebumen|   Elektronik-304| 2.9745E8|   4|
|  Kebumen|     Olahraga-321|  2.973E8|   5|
| Magelang|    Olahraga-1250|3.57975E8|   1|
| Magelang|     Makanan-2887| 3.0975E8|   2|
| Magelang|  Elektronik-2125| 3.0855E8|   3|
| Magelang|     Fashion-2006| 3.0765E8|   4|
| Magelang|    Otomotif-4854|3.05025E8|   5|
|Purworejo|       Makanan-68| 3.3195E8|   1|
|Purworejo|    Otomotif-1072| 3.3165E8|   2|
|Purworejo|Rumah Tangga-4823|  3.282E8|   3|
|Purworejo|Rumah Tangga-1593| 3.1455E8|   4|
|Purworejo| Rumah Tangga-277|3.09375E8|   5|
+---------+-----------------+---------+----+
only showing top 15 rows



In [9]:
# D. Running Total & Moving Average 7 Hari
df_harian = df_join.withColumn("tgl", F.to_date("tanggal_ts")) \
    .groupBy("kategori", "tgl").agg(F.sum("total_nilai").alias("omzet_harian"))
w_cum = Window.partitionBy("kategori").orderBy("tgl").rowsBetween(Window.unboundedPreceding, 0)
w_ma7 = Window.partitionBy("kategori").orderBy("tgl").rowsBetween(-6, 0)

df_harian.withColumn("running_total", F.sum("omzet_harian").over(w_cum)) \
         .withColumn("ma_7hari", F.avg("omzet_harian").over(w_ma7)).show(10)

[Stage 65:=============================>                            (1 + 1) / 2]

+--------+----------+------------+-------------+--------------------+
|kategori|       tgl|omzet_harian|running_total|            ma_7hari|
+--------+----------+------------+-------------+--------------------+
|    Buku|2026-01-01|  1.012666E9|   1.012666E9|          1.012666E9|
|    Buku|2026-01-02| 1.1973155E9|  2.2099815E9|        1.10499075E9|
|    Buku|2026-01-03|1.12902925E9| 3.33901075E9|1.1130035833333333E9|
|    Buku|2026-01-04|1.12665275E9|  4.4656635E9|       1.116415875E9|
|    Buku|2026-01-05|1.15962175E9| 5.62528525E9|        1.12505705E9|
|    Buku|2026-01-06|1.17997025E9|  6.8052555E9|        1.13420925E9|
|    Buku|2026-01-07|1.10306975E9| 7.90832525E9|        1.12976075E9|
|    Buku|2026-01-08| 1.2315195E9| 9.13984475E9|1.1610255357142856E9|
|    Buku|2026-01-09|1.25491775E9|1.03947625E10|1.1692544285714285E9|
|    Buku|2026-01-10|   1.10175E9|1.14965125E10| 1.165357392857143E9|
+--------+----------+------------+-------------+--------------------+
only showing top 10 

In [10]:
# E. Spark SQL & Perbandingan explain()
df_selesai.createOrReplaceTempView("v_selesai")
sql_b = spark.sql("""
    WITH b AS (SELECT bulan, SUM(total_nilai) AS omzet FROM v_selesai GROUP BY bulan)
    SELECT bulan, omzet, LAG(omzet) OVER (ORDER BY bulan) AS omzet_prev,
           ROUND((omzet - LAG(omzet) OVER (ORDER BY bulan)) / LAG(omzet) OVER (ORDER BY bulan) * 100, 2) AS growth_pct
    FROM b ORDER BY bulan
""")
sql_b.show()
sql_b.explain()

26/11/28 12:26:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:26:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:26:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:26:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:26:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:26:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 1

+-------+---------------+---------------+----------+
|  bulan|          omzet|     omzet_prev|growth_pct|
+-------+---------------+---------------+----------+
|2026-01|3.0074566725E11|           NULL|      NULL|
|2026-02|2.7197526675E11|3.0074566725E11|     -9.57|
|2026-03|3.0127041575E11|2.7197526675E11|     10.77|
|2026-04|2.9058154625E11|3.0127041575E11|     -3.55|
|2026-05|  3.00722052E11|2.9058154625E11|      3.49|
|2026-06|2.9155883375E11|  3.00722052E11|     -3.05|
+-------+---------------+---------------+----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [bulan#405, omzet#766, omzet_prev#764, round((((omzet#766 - _we1#768) / _we2#769) * 100.0), 2) AS growth_pct#765]
   +- Window [lag(omzet#766, -1, null) windowspecdefinition(bulan#405 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -1, -1)) AS omzet_prev#764, lag(omzet#766, -1, null) windowspecdefinition(bulan#405 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -1, -1)) AS _we1#768, lag(omzet#766, -

26/11/28 12:26:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:26:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [11]:
# F. RFM & Prinsip Pareto (80% Omzet)
max_date = df_selesai.agg(F.max("tanggal_ts")).collect()[0][0]
df_rfm = df_selesai.groupBy("pelanggan_id").agg(
    F.datediff(F.lit(max_date), F.max("tanggal_ts")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("total_nilai").alias("monetary")
)

tot_omzet = df_rfm.agg(F.sum("monetary")).collect()[0][0]
tot_pel = df_rfm.count()
w_par = Window.orderBy(F.desc("monetary")).rowsBetween(Window.unboundedPreceding, 0)
pel_80 = df_rfm.withColumn("cum_pct", F.sum("monetary").over(w_par) / tot_omzet * 100) \
               .filter(F.col("cum_pct") <= 80).count()
print(f"Pelanggan penyumbang 80% omzet: {pel_80:,} dari {tot_pel:,} ({pel_80/tot_pel*100:.2f}%)")

26/11/28 12:30:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:30:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:30:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:30:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:30:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/28 12:30:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Pelanggan penyumbang 80% omzet: 16,928 dari 49,063 (34.50%)


In [12]:
# G. Perbandingan Perilaku Antar Membership
df_trx.join(F.broadcast(df_pelanggan.select("pelanggan_id", "membership")), "pelanggan_id") \
      .groupBy("membership").agg(
          F.round(F.avg("total_nilai"), 0).alias("avg_order_value"),
          F.count("order_id").alias("frekuensi_total"),
          F.round(F.avg("flag_batal") * 100, 2).alias("tingkat_batal_pct")
      ).show()

[Stage 99:>                                                         (0 + 2) / 2]

+----------+---------------+---------------+-----------------+
|membership|avg_order_value|frekuensi_total|tingkat_batal_pct|
+----------+---------------+---------------+-----------------+
|      Gold|      1158320.0|         582892|            19.82|
|    Silver|      1163168.0|        1202830|            19.72|
|  Platinum|      1159402.0|         201285|            19.79|
+----------+---------------+---------------+-----------------+



In [13]:
# H. Identifikasi Pelanggan Berisiko Pergi (Churn > 30 Hari)
df_churn = df_rfm.filter(F.col("recency") > 30).join(F.broadcast(df_pelanggan), "pelanggan_id")
print(f"Jumlah pelanggan berisiko pergi (tidak transaksi >30 hari): {df_churn.count():,}")
df_churn.groupBy("membership", "kota").count().orderBy(F.desc("count")).show(10)

Jumlah pelanggan berisiko pergi (tidak transaksi >30 hari): 8,583


+----------+----------+-----+
|membership|      kota|count|
+----------+----------+-----+
|    Silver|      Solo|  686|
|    Silver|   Kebumen|  686|
|    Silver|  Wonosobo|  664|
|    Silver| Purworejo|  659|
|    Silver|  Magelang|  658|
|    Silver|  Semarang|  631|
|    Silver|Yogyakarta|  618|
|    Silver|Temanggung|  614|
|      Gold|      Solo|  372|
|      Gold|  Magelang|  318|
+----------+----------+-----+
only showing top 10 rows



## Tahap 5 : Penyimpanan NoSQL (MongoDB)
• Muat pelanggan.json dan produk.json ke MongoDB (database tidarmart, koleksi pelanggan dan produk).
Jelaskan mengapa produk dengan atribut bervariasi cocok disimpan sebagai dokumen. Pada PyMongo,
pastikan tipe data Python biasa (int, float, str, list), bukan tipe NumPy.

• Buat ringkasan “profil pelanggan 360” hasil Spark (jumlah pesanan, total belanja, tingkat pembatalan,
kategori favorit) dan simpan ke koleksi profil_pelanggan, ditulis secara batch (insert_many per potongan,
misalnya 5.000 dokumen), bukan satu per satu dan bukan seluruhnya sekaligus.

• Buat indeks pada pelanggan_id dan tunjukkan perbedaan waktu/rencana eksekusi query dengan dan tanpa
indeks (explain).

• Tunjukkan minimal lima operasi: query dengan operator ($gt, $in), query pada field array (tags), query
pada field yang tidak selalu ada ($exists), update satu dokumen, dan satu aggregation pipeline ($group /
$match / $sort).

• Jawab : pada kasus apa data sebaiknya tetap di Parquet/HDFS dan pada kasus apa sebaiknya di MongoDB?
Berikan minimal dua skenario penggunaan.

In [14]:
# Memuat pelanggan.json & produk.json ke MongoDB
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["tidarmart"]

def load_jsonl(path, col_name, batch_size=5000):
    col = db[col_name]; col.drop(); batch = []
    with open(path) as f:
        for line in f:
            batch.append(json.loads(line))
            if len(batch) >= batch_size:
                col.insert_many(batch); batch.clear()
        if batch: col.insert_many(batch)

load_jsonl("data_proyek/pelanggan.json", "pelanggan")
load_jsonl("data_proyek/produk.json", "produk")
print("Pelanggan:", db.pelanggan.count_documents({}), "| Produk:", db.produk.count_documents({}))

Pelanggan: 50000 | Produk: 5000


In [15]:
# Membuat Profil Pelanggan 360 & Tulis Batch ke MongoDB
w_fav = Window.partitionBy("pelanggan_id").orderBy(F.desc("cnt"))
fav_kat = (df_join.groupBy("pelanggan_id", "kategori").count().withColumnRenamed("count", "cnt")
           .withColumn("rn", F.row_number().over(w_fav)).filter(F.col("rn") == 1)
           .select("pelanggan_id", F.col("kategori").alias("kategori_favorit")))

df_p360 = (df_trx.groupBy("pelanggan_id").agg(
    F.count("order_id").alias("jumlah_pesanan"),
    F.sum("total_nilai").alias("total_belanja"),
    F.avg("flag_batal").alias("tingkat_pembatalan")
).join(fav_kat, "pelanggan_id", "left"))

col_p = db["profil_pelanggan"]; col_p.drop(); batch = []
for r in df_p360.toLocalIterator():
    batch.append({
        "pelanggan_id": int(r["pelanggan_id"]), "jumlah_pesanan": int(r["jumlah_pesanan"]),
        "total_belanja": float(r["total_belanja"]), "tingkat_pembatalan": float(r["tingkat_pembatalan"]),
        "kategori_favorit": str(r["kategori_favorit"]) if r["kategori_favorit"] else None
    })
    if len(batch) >= 5000:
        col_p.insert_many(batch); batch.clear()
if batch: col_p.insert_many(batch)
print("Total Profil 360:", col_p.count_documents({}))

[Stage 126:>                                                        (0 + 1) / 1]

Total Profil 360: 49403


In [16]:
# Pembuatan Indeks & Perbandingan explain() 
exp_no_idx = db.profil_pelanggan.find({"pelanggan_id": 100}).explain()["executionStats"]
db.profil_pelanggan.create_index("pelanggan_id")
exp_idx = db.profil_pelanggan.find({"pelanggan_id": 100}).explain()["executionStats"]

print(f"Tanpa Indeks -> Docs Examined: {exp_no_idx['totalDocsExamined']} | Waktu: {exp_no_idx['executionTimeMillis']} ms")
print(f"Dengan Indeks -> Docs Examined: {exp_idx['totalDocsExamined']} | Waktu: {exp_idx['executionTimeMillis']} ms")

Tanpa Indeks -> Docs Examined: 49403 | Waktu: 54 ms
Dengan Indeks -> Docs Examined: 1 | Waktu: 15 ms


In [17]:
# Lima Operasi Wajib MongoDB
print("1. $gt & $in:", db.profil_pelanggan.count_documents({"total_belanja": {"$gt": 1e7}, "kategori_favorit": {"$in": ["Elektronik", "Otomotif"]}}))
print("2. Array tags ('promo'):", db.produk.count_documents({"tags": "promo"}))
print("3. Field opsional ($exists garansi_bulan):", db.produk.count_documents({"garansi_bulan": {"$exists": True}}))
db.pelanggan.update_one({"pelanggan_id": 1}, {"$set": {"membership": "Platinum"}})
print("4. Update pelanggan_id=1 selesai.")
agg = list(db.produk.aggregate([
    {"$match": {"harga_dasar": {"$gt": 100000}}},
    {"$group": {"_id": "$kategori", "avg_harga": {"$avg": "$harga_dasar"}}},
    {"$sort": {"avg_harga": -1}}
]))
print("5. Aggregation Pipeline (Top 3):", agg[:3])

1. $gt & $in: 7656
2. Array tags ('promo'): 604
3. Field opsional ($exists garansi_bulan): 626
4. Update pelanggan_id=1 selesai.
5. Aggregation Pipeline (Top 3): [{'_id': 'Olahraga', 'avg_harga': 599736.8421052631}, {'_id': 'Makanan', 'avg_harga': 597827.4760383386}, {'_id': 'Rumah Tangga', 'avg_harga': 595000.0}]


## Tahap 6 : Pemrosesan Real-Time (Structured Streaming)
• Alirkan berkas pada stream_input/ menggunakan readStream dengan skema eksplisit dan
maxFilesPerTrigger = 1 (atau 2). Karena data mengalir dari HDFS atau folder lokal, jelaskan pilihan Anda.

• Hitung agregasi berbasis window waktu (misalnya jumlah pesanan dan omzet per jendela 1 jam per kota)
dan tampilkan hasil yang terus diperbarui melalui memory sink atau file sink.

• Bangun aturan deteksi anomali real-time sederhana: tandai transaksi dengan total_nilai di atas ambang
yang Anda tentukan secara statistik dari data batch (misalnya persentil 99,9) dan tulis ke sink terpisah.

• Saat berkas stream baru ditambahkan ke folder, buktikan bahwa hasil bertambah (tangkap hasil sebelum
dan sesudah).

• Hentikan seluruh query streaming dengan benar (query.stop()) dan jelaskan mode keluaran (append /
update / complete) yang Anda pilih beserta alasannya.

In [18]:
# Skema Eksplisit, Ambang Anomali, dan Setup readStream
ambang_anomali = df_trx.approxQuantile("total_nilai", [0.999], 0.001)[0]
print(f"Ambang batas anomali (P99.9): Rp {ambang_anomali:,.0f}")

stream_schema = StructType([
    StructField("order_id", StringType()), StructField("tanggal", TimestampType()),
    StructField("pelanggan_id", IntegerType()), StructField("produk_id", IntegerType()),
    StructField("qty", DoubleType()), StructField("harga_satuan", DoubleType()),
    StructField("diskon_persen", DoubleType()), StructField("metode_pembayaran", StringType()),
    StructField("kota", StringType()), StructField("berat_kg", DoubleType()),
    StructField("jarak_km", DoubleType()), StructField("lama_kirim_hari", IntegerType()),
    StructField("status_pesanan", StringType())
])

df_stream = (spark.readStream.option("header", True).option("maxFilesPerTrigger", 1)
    .schema(stream_schema).csv("data_proyek/stream_input/")
    .dropna(subset=["qty", "harga_satuan"])
    .withColumn("kota", F.initcap(F.trim(F.col("kota"))))
    .withColumn("total_nilai", F.col("qty") * F.abs(F.col("harga_satuan")) * (1 - F.col("diskon_persen") / 100.0)))

Ambang batas anomali (P99.9): Rp 10,500,000


In [19]:
# Menjalankan Query Window 1 Jam & Deteksi Anomali
q_agg = (df_stream.groupBy(F.window("tanggal", "1 hour"), "kota")
    .agg(F.count("order_id").alias("jml_pesanan"), F.sum("total_nilai").alias("omzet"))
    .writeStream.outputMode("complete").format("memory").queryName("stream_omzet").start())

q_anom = (df_stream.filter(F.col("total_nilai") > ambang_anomali)
    .select("order_id", "kota", "total_nilai")
    .writeStream.outputMode("append").format("memory").queryName("stream_anomali").start())

26/11/28 12:45:43 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3e93a2ad-52f2-4203-adef-23745d4b522b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/28 12:45:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/28 12:45:44 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e4ca04b1-ebf5-43c2-9d45-488ee826f488. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/28 12:45:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not support

In [20]:
# Memantau Pertambahan Data & Hentikan Streaming
time.sleep(5)
print("=== Snapshot 1 ===")
spark.sql("SELECT COUNT(*) AS baris_agg FROM stream_omzet").show()
spark.sql("SELECT COUNT(*) AS jml_anomali FROM stream_anomali").show()

time.sleep(10)
print("=== Snapshot 2 (Setelah micro-batch tambahan masuk) ===")
spark.sql("SELECT * FROM stream_omzet LIMIT 5").show(truncate=False)
spark.sql("SELECT COUNT(*) AS jml_anomali FROM stream_anomali").show()

q_agg.stop(); q_anom.stop()
print("Seluruh query streaming berhasil dihentikan.")

=== Snapshot 1 ===
+---------+
|baris_agg|
+---------+
|      192|
+---------+

+-----------+
|jml_anomali|
+-----------+
|         55|
+-----------+

=== Snapshot 2 (Setelah micro-batch tambahan masuk) ===
+------------------------------------------+----------+-----------+-----------+
|window                                    |kota      |jml_pesanan|omzet      |
+------------------------------------------+----------+-----------+-----------+
|{2026-07-01 02:00:00, 2026-07-01 03:00:00}|Magelang  |507        |6.56238E8  |
|{2026-07-01 04:00:00, 2026-07-01 05:00:00}|Magelang  |483        |5.4749275E8|
|{2026-07-01 11:00:00, 2026-07-01 12:00:00}|Purworejo |531        |6.0251625E8|
|{2026-07-01 01:00:00, 2026-07-01 02:00:00}|Temanggung|515        |6.5295525E8|
|{2026-07-01 15:00:00, 2026-07-01 16:00:00}|Yogyakarta|472        |5.42105E8  |
+------------------------------------------+----------+-----------+-----------+

+-----------+
|jml_anomali|
+-----------+
|         55|
+-----------+

S

## Tahap 7 : Machine Learning dengan MLlib
• Model 1 – Klasifikasi pembatalan pesanan. Target: apakah pesanan Dibatalkan (abaikan status
Dikembalikan atau jelaskan perlakuan Anda). Fitur minimal: metode_pembayaran (kategorikal), kota,
diskon_persen, jarak_km, berat_kg, lama_kirim_hari, qty. Latih dan bandingkan LogisticRegression dan
RandomForestClassifier. Laporkan AUC, Accuracy, F1, dan matriks konfusi. Karena kelas tidak seimbang
(sekitar 20% batal), jelaskan mengapa Accuracy saja menyesatkan dan lakukan penanganan (misalnya
class weight atau pembahasan ambang).

• Model 2 – Regresi. Prediksi lama_kirim_hari atau total_nilai (pilih salah satu dan jelaskan manfaat
bisnisnya) menggunakan LinearRegression dan satu model pembanding. Laporkan RMSE dan R². Hati-
hati terhadap kebocoran data (data leakage): jangan memasukkan fitur yang merupakan turunan langsung
dari target.

• Model 3 – Segmentasi pelanggan (K-Means). Bangun fitur RFM per pelanggan (atau fitur perilaku lain),
lakukan StandardScaler, pilih jumlah klaster k dengan evaluasi (Silhouette dan/atau elbow pada k = 2
sampai 8), dan beri nama/profil bisnis untuk tiap klaster (misalnya “pelanggan setia bernilai tinggi”).

• Lakukan satu eksperimen peningkatan (feature engineering atau penyetelan parameter dengan beberapa
nilai) dan tunjukkan dampaknya terhadap metrik dalam tabel. Hasil yang tidak membaik tetap sah selama
dianalisis jujur dan beralasan.

• Simpan setiap PipelineModel ke HDFS (/tidarmart/models/...) dan tunjukkan bahwa model dapat dimuat
kembali (PipelineModel.load) menghasilkan prediksi yang sama.

In [21]:
# Model 1 (Klasifikasi): Persiapan Data & Class Weight
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, ClusteringEvaluator

df_ml = df_trx.filter(F.col("status_pesanan") != "Dikembalikan").sample(fraction=0.2, seed=SEED)
rasio_batal = df_ml.filter(F.col("flag_batal") == 1).count() / df_ml.count()
df_ml = df_ml.withColumn("weight", F.when(F.col("flag_batal") == 1, 1 - rasio_batal).otherwise(rasio_batal))

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=SEED)

In [22]:
# Model 1: Melatih & Evaluasi (LR vs RF + Confusion Matrix)
idx_m = StringIndexer(inputCol="metode_pembayaran", outputCol="m_idx", handleInvalid="keep")
idx_k = StringIndexer(inputCol="kota", outputCol="k_idx", handleInvalid="keep")
ohe = OneHotEncoder(inputCols=["m_idx", "k_idx"], outputCols=["m_vec", "k_vec"])
asm = VectorAssembler(inputCols=["m_vec", "k_vec", "diskon_persen", "jarak_km", "berat_kg", "lama_kirim_hari", "qty"], outputCol="features")

lr = LogisticRegression(labelCol="flag_batal", featuresCol="features", weightCol="weight")
rf = RandomForestClassifier(labelCol="flag_batal", featuresCol="features", weightCol="weight", numTrees=25, maxDepth=6, seed=SEED)

pipe_lr = Pipeline(stages=[idx_m, idx_k, ohe, asm, lr]).fit(train_df)
pipe_rf = Pipeline(stages=[idx_m, idx_k, ohe, asm, rf]).fit(train_df)

e_auc = BinaryClassificationEvaluator(labelCol="flag_batal", metricName="areaUnderROC")
e_acc = MulticlassClassificationEvaluator(labelCol="flag_batal", metricName="accuracy")
e_f1  = MulticlassClassificationEvaluator(labelCol="flag_batal", metricName="f1")

for nama, m in [("LogisticRegression", pipe_lr), ("RandomForest (depth=6)", pipe_rf)]:
    p = m.transform(test_df)
    print(f"{nama} -> AUC: {e_auc.evaluate(p):.4f} | Acc: {e_acc.evaluate(p):.4f} | F1: {e_f1.evaluate(p):.4f}")
    p.groupBy("flag_batal", "prediction").count().show() # Confusion Matrix

LogisticRegression -> AUC: 0.6123 | Acc: 0.6023 | F1: 0.6396


+----------+----------+-----+
|flag_batal|prediction|count|
+----------+----------+-----+
|         1|       1.0| 8613|
|         0|       0.0|38014|
|         1|       0.0| 7207|
|         0|       1.0|23584|
+----------+----------+-----+



RandomForest (depth=6) -> AUC: 0.6092 | Acc: 0.5765 | F1: 0.6175


[Stage 289:>                                                        (0 + 2) / 2]

+----------+----------+-----+
|flag_batal|prediction|count|
+----------+----------+-----+
|         1|       1.0| 9238|
|         0|       0.0|35395|
|         1|       0.0| 6582|
|         0|       1.0|26203|
+----------+----------+-----+



In [23]:
# Model 1: Eksperimen Tuning, Simpan ke HDFS & Uji Muat Ulang
rf_tuned = RandomForestClassifier(labelCol="flag_batal", featuresCol="features", weightCol="weight", numTrees=45, maxDepth=8, seed=SEED)
pipe_rf_tuned = Pipeline(stages=[idx_m, idx_k, ohe, asm, rf_tuned]).fit(train_df)
p_tuned = pipe_rf_tuned.transform(test_df)
print(f"RF Tuned (45, 8) -> AUC: {e_auc.evaluate(p_tuned):.4f} | F1: {e_f1.evaluate(p_tuned):.4f}")

# Simpan ke HDFS & buktikan prediksi identik setelah load
pipe_rf_tuned.write().overwrite().save(f"{HDFS_BASE}/models/rf_pembatalan")
loaded_rf = PipelineModel.load(f"{HDFS_BASE}/models/rf_pembatalan")
print("AUC Loaded Model:", round(e_auc.evaluate(loaded_rf.transform(test_df)), 4))

26/11/28 13:05:12 WARN DAGScheduler: Broadcasting large task binary with size 1675.6 KiB
26/11/28 13:05:21 WARN DAGScheduler: Broadcasting large task binary with size 1056.0 KiB
26/11/28 13:05:24 WARN DAGScheduler: Broadcasting large task binary with size 1067.8 KiB
                                                                                

RF Tuned (45, 8) -> AUC: 0.6103 | F1: 0.6300


26/11/28 13:05:42 WARN DAGScheduler: Broadcasting large task binary with size 1101.8 KiB
                                                                                

AUC Loaded Model: 0.6103


In [24]:
# Model 3: Segmentasi Pelanggan RFM (K-Means)
rfm_asm = VectorAssembler(inputCols=["recency", "frequency", "monetary"], outputCol="rfm_raw")
scaler = StandardScaler(inputCol="rfm_raw", outputCol="rfm_scaled", withStd=True, withMean=True)
e_sil = ClusteringEvaluator(featuresCol="rfm_scaled")

for k in range(2, 9):
    km_pipe = Pipeline(stages=[rfm_asm, scaler, KMeans(featuresCol="rfm_scaled", k=k, seed=SEED)]).fit(df_rfm)
    print(f"K={k} -> Silhouette: {e_sil.evaluate(km_pipe.transform(df_rfm)):.4f}")

# Latih model final (misal K=4 terbaik), profilkan klaster, dan simpan ke HDFS
km_final = Pipeline(stages=[rfm_asm, scaler, KMeans(featuresCol="rfm_scaled", k=4, seed=SEED)]).fit(df_rfm)
km_final.transform(df_rfm).groupBy("prediction").agg(
    F.count("*").alias("jml_pelanggan"), F.round(F.avg("recency"), 1).alias("avg_rec"),
    F.round(F.avg("frequency"), 1).alias("avg_freq"), F.round(F.avg("monetary"), 0).alias("avg_mon")
).orderBy("prediction").show()
km_final.write().overwrite().save(f"{HDFS_BASE}/models/kmeans_rfm")

K=2 -> Silhouette: 0.8945


K=3 -> Silhouette: 0.8867


K=4 -> Silhouette: 0.8083


K=5 -> Silhouette: 0.6956


K=6 -> Silhouette: 0.7117


K=7 -> Silhouette: 0.6958


K=8 -> Silhouette: 0.6534


+----------+-------------+-------+--------+------------+
|prediction|jml_pelanggan|avg_rec|avg_freq|     avg_mon|
+----------+-------------+-------+--------+------------+
|         0|        41805|    9.9|    26.1| 2.9781629E7|
|         1|         5450|   84.3|     3.6|   4095016.0|
|         2|         1807|    0.5|   231.5|2.68095047E8|
|         3|            1|    0.0|  4637.0| 5.0672045E9|
+----------+-------------+-------+--------+------------+



## Tahap 8 : Integrasi, Optimasi Sumber Daya, dan Penutup
• Integrasi batch dan streaming: muat model klasifikasi pembatalan dari HDFS, terapkan pada data
stream sehingga setiap transaksi baru memperoleh skor risiko pembatalan, dan tulis pesanan berisiko
tinggi ke sink terpisah. Ini adalah bagian yang menyatukan seluruh pipeline.

• Studi sumber daya: jalankan satu pekerjaan berat (misalnya join + agregasi seluruh data) dengan dua
konfigurasi berbeda (misalnya shuffle.partitions = 4 vs 200, atau cache vs tanpa cache) dan sajikan tabel
waktu eksekusi. Lampirkan tangkapan layar Spark UI (port 4040) yang menunjukkan stage/task dan
jelaskan satu temuan Anda.

• Catatan masalah (troubleshooting log): tuliskan minimal tiga masalah nyata yang Anda temui (error,
kehabisan memori, hasil tak terduga) beserta cara menyelesaikannya.

• Rekomendasi: tuliskan tiga rekomendasi bisnis berbasis bukti dan satu rencana peningkatan skala
(bagaimana arsitektur Anda bila data menjadi 1.000× lebih besar di klaster sungguhan: komponen apa
yang berubah dan mengapa).

In [25]:
# Integrasi Batch–Streaming (Skor Risiko Pembatalan Real-Time)
model_batal = PipelineModel.load(f"{HDFS_BASE}/models/rf_pembatalan")
risiko_tinggi = (model_batal.transform(df_stream)
    .filter(F.col("prediction") == 1.0)
    .select("order_id", "kota", "metode_pembayaran", "lama_kirim_hari", "prediction"))

q_ml = risiko_tinggi.writeStream.outputMode("append").format("memory").queryName("stream_risiko").start()
time.sleep(10)
spark.sql("SELECT * FROM stream_risiko LIMIT 10").show()
q_ml.stop()

26/11/28 13:12:29 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6f611de5-10b0-4b4a-8df7-d103293064d9. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/28 13:12:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/28 13:12:29 WARN DAGScheduler: Broadcasting large task binary with size 1075.3 KiB
26/11/28 13:12:30 WARN DAGScheduler: Broadcasting large task binary with size 1075.3 KiB
26/11/28 13:12:31 WARN DAGScheduler: Broadcasting large task binary with size 1075.3 KiB
26/11/28 13:12:31 WARN DAGScheduler: Broadcasting large task binary with size 1075.3 KiB
26/11/28 13:12:32 WARN DAGScheduler: Broadcasting large task binary with size 1075.3 KiB
26/11/28 13:12:33 WARN DAG

+----------+----------+-----------------+---------------+----------+
|  order_id|      kota|metode_pembayaran|lama_kirim_hari|prediction|
+----------+----------+-----------------+---------------+----------+
|ORD9010001|      Solo|    Transfer Bank|              4|       1.0|
|ORD9010005|  Semarang|     Kartu Kredit|              5|       1.0|
|ORD9010007|  Semarang|    Transfer Bank|              4|       1.0|
|ORD9010009|  Semarang|         E-Wallet|              5|       1.0|
|ORD9010010| Purworejo|         E-Wallet|              5|       1.0|
|ORD9010012|   Kebumen|    Transfer Bank|              4|       1.0|
|ORD9010014| Purworejo|              COD|              3|       1.0|
|ORD9010015|Temanggung|    Transfer Bank|              4|       1.0|
|ORD9010020| Purworejo|     Kartu Kredit|              5|       1.0|
|ORD9010022|  Semarang|     Kartu Kredit|              6|       1.0|
+----------+----------+-----------------+---------------+----------+



26/11/28 13:12:39 WARN DAGScheduler: Broadcasting large task binary with size 1075.3 KiB
26/11/28 13:12:39 WARN TaskSetManager: Stage 1269 contains a task of very large size (1953 KiB). The maximum recommended task size is 1000 KiB.
26/11/28 13:12:39 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 18, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@4210899d] is aborting.
26/11/28 13:12:39 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 18, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@4210899d] aborted.
26/11/28 13:12:39 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:267)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:130)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalys

In [26]:
# Studi Sumber Daya (shuffle.partitions 4 vs 200)

for p in ["4", "200"]:
    spark.conf.set("spark.sql.shuffle.partitions", p)
    t0 = time.time()
    df_trx.join(df_pelanggan, "pelanggan_id").groupBy("membership").agg(F.sum("total_nilai")).collect()
    print(f"shuffle.partitions = {p} -> Waktu eksekusi: {time.time() - t0:.2f} detik")

spark.conf.set("spark.sql.shuffle.partitions", "4")

shuffle.partitions = 4 -> Waktu eksekusi: 1.75 detik
shuffle.partitions = 200 -> Waktu eksekusi: 1.10 detik
